# MLflow cho Machine Learning truyền thống

Notebook này dành riêng cho bài toán Machine Learning kiểu cổ điển: hồi quy, phân loại, feature engineering, benchmark nhiều thuật toán và chọn mô hình tốt nhất.

Phạm vi của notebook:
- Tập trung vào workflow ML truyền thống.
- Dùng `scikit-learn` để minh họa vì gọn, dễ chạy và phổ biến.
- Pattern logging trong notebook này cũng áp dụng tương tự cho `XGBoost`, `LightGBM`, `CatBoost` hoặc pipeline tự viết.

Các chủ đề Deep Learning, Hugging Face, Transformer, LLM và AI Agent đã được tách sang notebook khác để nội dung rõ ràng hơn.

## 1. Khi nào nên dùng MLflow trong classical ML?

MLflow đặc biệt hữu ích khi bạn cần:
- So sánh nhiều mô hình trên cùng một dataset.
- Theo dõi hyperparameter, metric và artifact theo từng lần chạy.
- Lưu mô hình tốt nhất để phục vụ inference hoặc triển khai.
- Tạo lịch sử thí nghiệm đủ rõ để có thể audit hoặc tái hiện lại.

Trong classical ML, một `run` thường tương ứng với:
- Một thuật toán cụ thể như `LinearRegression`, `RandomForest`, `XGBoost`.
- Hoặc một cấu hình hyperparameter của cùng một thuật toán.

Cấu trúc thường gặp:
1. Tạo experiment theo bài toán.
2. Tạo run cha cho cả đợt benchmark.
3. Tạo nested run cho từng mô hình.
4. Log params, metrics, artifacts, model.
5. Dùng `search_runs()` để chọn run tốt nhất.


In [ ]:
# Nếu máy chưa có thư viện, chạy một lần rồi restart kernel.
# %pip install -q mlflow scikit-learn pandas matplotlib


In [ ]:
from pathlib import Path
import json
import tempfile

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd()
MLRUNS_DIR = (BASE_DIR / "mlruns").resolve()
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)

tracking_uri = MLRUNS_DIR.as_uri()
mlflow.set_tracking_uri(tracking_uri)
experiment_name = "mlflow_classical_ml"
experiment = mlflow.set_experiment(experiment_name)
client = MlflowClient(tracking_uri=tracking_uri)

print(f"Tracking URI: {tracking_uri}")
print(f"Experiment: {experiment_name}")
print(f"Experiment ID: {experiment.experiment_id}")


## 2. Tạo dữ liệu và tập mô hình benchmark

Thay vì thử một mô hình duy nhất, notebook này benchmark nhiều thuật toán trên cùng dataset. Đây là tình huống rất thường gặp trong thực tế.

Ở đây ta thử 4 mô hình thuộc 3 nhóm khác nhau:
- `LinearRegression`: baseline tuyến tính.
- `Ridge`: tuyến tính có regularization.
- `RandomForestRegressor`: bagging tree ensemble.
- `GradientBoostingRegressor`: boosting tree ensemble.

Ý tưởng này có thể mở rộng trực tiếp sang `XGBoost`, `LightGBM` hoặc `CatBoost`.

In [ ]:
X, y = make_regression(
    n_samples=1200,
    n_features=10,
    n_informative=7,
    noise=18.0,
    random_state=42,
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)
y_series = pd.Series(y, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X_df,
    y_series,
    test_size=0.2,
    random_state=42,
)

model_recipes = {
    "LinearRegression": {
        "params": {"fit_intercept": True},
        "builder": lambda cfg: LinearRegression(**cfg),
    },
    "Ridge": {
        "params": {"alpha": 1.0, "fit_intercept": True},
        "builder": lambda cfg: Ridge(**cfg),
    },
    "RandomForestRegressor": {
        "params": {
            "n_estimators": 200,
            "max_depth": 8,
            "min_samples_split": 4,
            "random_state": 42,
        },
        "builder": lambda cfg: RandomForestRegressor(**cfg),
    },
    "GradientBoostingRegressor": {
        "params": {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "max_depth": 3,
            "random_state": 42,
        },
        "builder": lambda cfg: GradientBoostingRegressor(**cfg),
    },
}

display(X_train.head())
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print("Models:", list(model_recipes.keys()))


## 3. Benchmark nhiều mô hình bằng nested runs

Phần này là workflow trung tâm cho classical ML:
- Một run cha đại diện cho cả đợt benchmark.
- Mỗi nested run đại diện cho một thuật toán.
- Mỗi nested run sẽ log `params`, `metrics`, artifact và model riêng.

Cách tổ chức này giúp MLflow UI dễ đọc hơn rất nhiều khi số lần thử tăng lên.

In [ ]:
best_rmse = float("inf")
best_run_id = None

with mlflow.start_run(run_name="classical_ml_benchmark") as parent_run:
    mlflow.set_tags(
        {
            "domain": "classical_ml",
            "notebook": "MLflow/mlflow_ML.ipynb",
            "dataset_type": "synthetic_regression",
        }
    )
    mlflow.log_param("num_models", len(model_recipes))
    mlflow.log_param("test_size", 0.2)

    for model_name, recipe in model_recipes.items():
        with mlflow.start_run(run_name=model_name, nested=True) as child_run:
            params = recipe["params"]
            estimator = recipe["builder"](params)
            estimator.fit(X_train, y_train)

            train_pred = estimator.predict(X_train)
            test_pred = estimator.predict(X_test)

            metrics = {
                "train_rmse": mean_squared_error(y_train, train_pred) ** 0.5,
                "test_rmse": mean_squared_error(y_test, test_pred) ** 0.5,
                "test_mae": mean_absolute_error(y_test, test_pred),
                "test_r2": r2_score(y_test, test_pred),
            }

            mlflow.log_param("model_name", model_name)
            mlflow.log_params(params)
            mlflow.log_metrics(metrics)
            mlflow.set_tag("algorithm_family", model_name.replace("Regressor", ""))

            with tempfile.TemporaryDirectory() as tmp_dir:
                tmp_path = Path(tmp_dir)

                report = {
                    "model_name": model_name,
                    "params": params,
                    "metrics": {k: round(v, 4) for k, v in metrics.items()},
                    "sample_predictions": [round(float(v), 3) for v in test_pred[:5]],
                }
                report_path = tmp_path / "summary.json"
                report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
                mlflow.log_artifact(str(report_path), artifact_path="reports")

                fig, ax = plt.subplots(figsize=(6, 6))
                ax.scatter(y_test, test_pred, alpha=0.55)
                min_val = min(y_test.min(), test_pred.min())
                max_val = max(y_test.max(), test_pred.max())
                ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")
                ax.set_title(f"{model_name}: Actual vs Predicted")
                ax.set_xlabel("Actual")
                ax.set_ylabel("Predicted")
                fig.tight_layout()
                plot_path = tmp_path / "actual_vs_predicted.png"
                fig.savefig(plot_path, dpi=120)
                plt.close(fig)
                mlflow.log_artifact(str(plot_path), artifact_path="plots")

            signature = infer_signature(X_train, estimator.predict(X_train))
            mlflow.sklearn.log_model(
                sk_model=estimator,
                artifact_path="model",
                signature=signature,
                input_example=X_train.head(5),
            )

            if metrics["test_rmse"] < best_rmse:
                best_rmse = metrics["test_rmse"]
                best_run_id = child_run.info.run_id

    mlflow.log_metric("best_child_test_rmse", best_rmse)

print(f"Best run ID: {best_run_id}")
print(f"Best RMSE: {best_rmse:.4f}")


## 4. So sánh run và load lại mô hình tốt nhất

Sau khi benchmark xong, bước tiếp theo thường là:
- Lấy bảng tổng hợp các run bằng `search_runs()`.
- Sắp xếp theo metric quan trọng nhất.
- Load model tốt nhất từ `run_id` để dùng lại.


In [ ]:
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_rmse ASC"],
)

columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.test_rmse",
    "metrics.test_r2",
    "params.model_name",
    "tags.mlflow.parentRunId",
]
display(runs_df[[col for col in columns if col in runs_df.columns]].head(10))

best_model_uri = f"runs:/{best_run_id}/model"
best_model = mlflow.pyfunc.load_model(best_model_uri)
preview = pd.DataFrame(
    {
        "actual": y_test.head(5).to_list(),
        "predicted": best_model.predict(X_test.head(5)).round(3),
    }
)
display(preview)

best_run = client.get_run(best_run_id)
print("Best artifact URI:", best_run.info.artifact_uri)
print("Best params:", best_run.data.params)
print("Best metrics:", best_run.data.metrics)


## 5. Mở rộng sang pipeline thực tế

Trong dự án thực tế, bạn thường nên log thêm:
- phiên bản dữ liệu,
- commit hash của code,
- feature list,
- output của bước tiền xử lý,
- confusion matrix hoặc residual analysis,
- file cấu hình train.

Nếu dùng framework khác ngoài `scikit-learn`, pattern vẫn giống nhau:
- `XGBoost` hoặc `LightGBM`: log params, metrics, feature importance, booster/model file.
- Pipeline custom: log artifact như `preprocessor.pkl`, schema dữ liệu, config YAML.

Chạy giao diện local để xem run:

```bash
cd /home/ubuntu/DataScience/MLOps-VNPost
mlflow ui --backend-store-uri ./mlruns --port 5000
```

Truy cập `http://127.0.0.1:5000` để so sánh các run trực quan.